# 01 · Exploratory analysis (development split only)

The held-out test rows (20%, fixed by `bnpl_risk.data.make_split`) are **excluded** from everything in this
notebook. The original EDA used all 10,000 rows, which let test rows inform modelling choices.

Questions:
1. Is the data clean, and is missingness informative?
2. Which inputs separate the risk tiers, and how?
3. How "real" is the target? A shallow tree tells us whether the labels follow a simple rule.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from bnpl_risk.config import (CATEGORICAL_COLS, CLASSES, NUMERIC_COLS, REPORTS_DIR,
                              MODELS_DIR, SPLIT_PATH, TARGET_COL)
from bnpl_risk.reporting import SERIES, _style
from bnpl_risk.seed import seed_everything

seed_everything()
_style()
pd.set_option("display.precision", 4)
from bnpl_risk.data import load_dev_frame

if not SPLIT_PATH.exists():
    raise FileNotFoundError("Run `python -m bnpl_risk.train` first: it writes the split indices.")
dev = load_dev_frame()
dev[TARGET_COL] = pd.Categorical(dev[TARGET_COL], categories=CLASSES, ordered=True)
print(f"development rows: {len(dev):,}")
dev.head()

development rows: 8,000


,Customer_ID,Age,Employment_Status,Income_USD,Credit_Score,Total_BNPL_Active_Loans,Total_BNPL_Debt_USD,Late_Payment_History,Shopping_Category_Most_Frequent,Average_Transaction_Value_USD,Default_Risk
0,C00001,29,Employed,73405.0,637.0,2,193,No,Fashion,118,Low
1,C00002,64,Employed,53732.0,815.0,0,0,No,Electronics,662,Low
2,C00006,20,Student,19759.0,488.0,3,513,Yes,Electronics,389,Medium
3,C00008,53,Freelancer,90887.0,730.0,4,773,No,Electronics,665,Low
4,C00009,42,Freelancer,51148.0,610.0,5,918,No,Electronics,681,Low


## 1. Data quality

In [2]:
quality = pd.DataFrame({
    "dtype": dev.dtypes.astype(str),
    "missing": dev.isna().sum(),
    "missing_%": (dev.isna().mean() * 100).round(2),
    "min": dev.min(numeric_only=True),
    "max": dev.max(numeric_only=True),
})
print(f"duplicate rows: {dev.duplicated().sum()}, duplicate IDs: {dev['Customer_ID'].duplicated().sum()}")
quality

duplicate rows: 0, duplicate IDs: 0


,dtype,missing,missing_%,min,max
Age,int64,0,0.00,18.0,64.0
Average_Transaction_Value_USD,int64,0,0.00,10.0,1420.0
Credit_Score,float64,235,2.94,300.0,850.0
Customer_ID,str,0,0.00,NaN,NaN
Default_Risk,category,0,0.00,NaN,NaN
Employment_Status,str,0,0.00,NaN,NaN
Income_USD,float64,252,3.15,5000.0,133107.0
Late_Payment_History,str,0,0.00,NaN,NaN
Shopping_Category_Most_Frequent,str,0,0.00,NaN,NaN
Total_BNPL_Active_Loans,int64,0,0.00,0.0,10.0


In [3]:
tier_share = dev[TARGET_COL].value_counts(normalize=True).reindex(CLASSES)
display(tier_share.to_frame("share").style.format("{:.1%}"))

# Is missingness informative? Missing rate by tier: flat rates mean missing completely at random.
miss = dev.groupby(TARGET_COL, observed=True)[["Income_USD", "Credit_Score"]].agg(lambda s: s.isna().mean())
miss.style.format("{:.2%}")

,share
Default_Risk,
Low,88.0%
Medium,6.8%
High,5.2%


,Income_USD,Credit_Score
Default_Risk,,
Low,3.18%,2.91%
Medium,2.77%,3.87%
High,3.12%,2.16%


Missingness is about 3% and flat across tiers (consistent with MCAR), so median imputation inside the pipeline is
adequate. The KNN-vs-median ablation in `reports/benchmark.md` confirms it: KNN imputation does not help.

## 2. Numeric inputs by tier

In [4]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
colors = ["#86b6ef", "#2a78d6", "#104281"]  # ordinal ramp Low -> High
for ax, col in zip(axes.flat, NUMERIC_COLS):
    data = [dev.loc[dev[TARGET_COL] == c, col].dropna() for c in CLASSES]
    bp = ax.boxplot(data, tick_labels=CLASSES, patch_artist=True, widths=0.5, showfliers=False,
                    medianprops={"color": "#fcfcfb", "linewidth": 2})
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_edgecolor(c)
    ax.set_title(col)
    ax.grid(axis="x", visible=False)
fig.tight_layout()

In [5]:
dev.groupby(TARGET_COL, observed=True)[NUMERIC_COLS].median()

,Age,Income_USD,Credit_Score,Total_BNPL_Active_Loans,Total_BNPL_Debt_USD,Average_Transaction_Value_USD
Default_Risk,,,,,,
Low,31.0,62940.0,682.0,2.0,228.0,299.0
Medium,31.0,15549.0,589.0,3.0,465.0,313.0
High,32.0,9789.0,572.0,4.0,692.0,310.0


Income and BNPL debt separate the tiers by an order of magnitude; age and purchase size barely move.
Income is **floored at exactly $5,000**, a censoring artifact of the generator:

In [6]:
floor = dev["Income_USD"] == dev["Income_USD"].min()
print(f"rows at the income floor ${dev['Income_USD'].min():,.0f}: {floor.sum()}")
dev.loc[floor, TARGET_COL].value_counts(normalize=True).reindex(CLASSES).to_frame("share at floor").style.format("{:.1%}")

rows at the income floor $5,000: 176


,share at floor
Default_Risk,
Low,25.0%
Medium,16.5%
High,58.5%


## 3. Categorical inputs

In [7]:
for col in CATEGORICAL_COLS:
    tab = pd.crosstab(dev[col], dev[TARGET_COL], normalize="index")
    display(Markdown(f"**{col}**"), tab.style.format("{:.1%}").background_gradient(cmap="Blues", axis=None))

**Employment_Status**

Default_Risk,Low,Medium,High
Employment_Status,,,
Employed,98.3%,1.4%,0.2%
Freelancer,92.3%,6.0%,1.7%
Student,66.3%,18.6%,15.1%
Unemployed,35.6%,27.2%,37.3%


**Late_Payment_History**

Default_Risk,Low,Medium,High
Late_Payment_History,,,
No,94.6%,3.0%,2.4%
Yes,68.0%,18.2%,13.8%


**Shopping_Category_Most_Frequent**

Default_Risk,Low,Medium,High
Shopping_Category_Most_Frequent,,,
Electronics,87.7%,7.0%,5.3%
Fashion,88.0%,6.8%,5.2%
Groceries/Essentials,89.4%,6.4%,4.2%
Home/Furniture,88.8%,6.2%,5.0%
Travel,86.9%,7.3%,5.8%


Employment status and late-payment history are strong signals; shopping category is flat (noise).
These are exactly the columns the original XGBoost baseline never received.

## 4. Leverage: the dominant driver

In [8]:
from bnpl_risk.features import engineer
from bnpl_risk.preprocessing import NumericImputer

imputed = NumericImputer().fit(dev).transform(dev)
feats = engineer(imputed).assign(tier=dev[TARGET_COL].to_numpy())
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.logspace(-4, 0, 60)
for c, color in zip(CLASSES, colors):
    ax.hist(feats.loc[feats.tier == c, "bnpl_dti"].clip(1e-4), bins=bins, color=color, alpha=0.85,
            label=c, density=True, histtype="stepfilled")
ax.set_xscale("log")
ax.set_xlabel("BNPL debt / annual income (log scale)")
ax.set_title("Debt-to-income by tier")
ax.legend()
feats.groupby("tier", observed=True)[["bnpl_dti", "ticket_to_monthly_income", "debt_per_loan"]].median()

,bnpl_dti,ticket_to_monthly_income,debt_per_loan
tier,,,
High,0.0658,0.3168,173.00
Low,0.0045,0.0618,140.00
Medium,0.0290,0.1511,162.25


## 5. How synthetic is the target?

If a depth-limited decision tree recovers the labels almost perfectly, the target is close to a hand-written rule.
That caps how much any model can tell us about real-world credit risk.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text
from bnpl_risk.data import encode_target

X = pd.concat([feats.drop(columns="tier"), imputed[NUMERIC_COLS],
               pd.get_dummies(dev[CATEGORICAL_COLS], dtype=float)], axis=1)
y = encode_target(dev[TARGET_COL].astype(str))
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for depth in (1, 2, 3, 4, 6):
    s = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=42), X, y, cv=cv, scoring="f1_macro")
    rows.append({"max_depth": depth, "macro_F1_mean": s.mean(), "macro_F1_sd": s.std()})
pd.DataFrame(rows)

,max_depth,macro_F1_mean,macro_F1_sd
0,1,0.5545,0.0197
1,2,0.7923,0.0274
2,3,0.8054,0.0203
3,4,0.8293,0.0218
4,6,0.8403,0.0140


In [10]:
print(export_text(DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y),
                  feature_names=list(X.columns), class_names=list(CLASSES)))

|--- bnpl_dti <= 0.03
|   |--- late_x_dti <= 0.01
|   |   |--- Employment_Status_Unemployed <= 0.50
|   |   |   |--- class: Low
|   |   |--- Employment_Status_Unemployed >  0.50
|   |   |   |--- class: Low
|   |--- late_x_dti >  0.01
|   |   |--- late_x_dti <= 0.02
|   |   |   |--- class: Low
|   |   |--- late_x_dti >  0.02
|   |   |   |--- class: Medium
|--- bnpl_dti >  0.03
|   |--- bnpl_dti <= 0.05
|   |   |--- late_x_dti <= 0.04
|   |   |   |--- class: Medium
|   |   |--- late_x_dti >  0.04
|   |   |   |--- class: High
|   |--- bnpl_dti >  0.05
|   |   |--- bnpl_dti <= 0.07
|   |   |   |--- class: High
|   |   |--- bnpl_dti >  0.07
|   |   |   |--- class: High



A depth-4 tree on one engineered ratio plus two flags already reaches the macro F1 the old README claimed
for the neural network. The labels are a near-deterministic function of **debt-to-income**, **late payments** and
**employment**. Consequences carried into the project:

* expect *all* reasonable models to score high and close together. Differences between families are noise
  unless a corrected test says otherwise;
* prefer the simplest well-calibrated model;
* report every number as a benchmark result on synthetic data, not as an estimate of real-world performance.

## 6. Correlation of numeric inputs

In [11]:
corr = imputed[NUMERIC_COLS].corr()
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=40, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "#0b0b0b")
fig.colorbar(im, shrink=0.8)
ax.set_title("Pearson correlation (dev split)")

Text(0.0, 1.0, 'Pearson correlation (dev split)')

Debt and active-loan count are strongly correlated (r ≈ 0.89). L2 regularisation in the logistic regression and
tree ensembles handle this without dropping either; `debt_per_loan` captures the part they do not share.